In [1]:
import torch
import warnings
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

from Model import *
from Utils import *
from Metrics import *
from LossUtils import *
from Preprocess import *
from Mmihcl import MMIHCL

plt.rcParams['figure.figsize'] = (6, 6)
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
# dataset_name including 'CITEseq_PBMC', 'ABseq_BMC', 'CODEX_tonsil', 'CITEseq_CyTOF_PBMC', 'CyTOF_human'
dataset_name = 'CITEseq_PBMC'

shared_adata_1, shared_adata_2, all_adata_1, all_adata_2 = readDataset(dataset_name)

print(shared_adata_1, shared_adata_2, all_adata_1, all_adata_2)

shared_adata_1, shared_adata_2, all_adata_1, all_adata_2 = preprocessAnnData(
    dataset_name=dataset_name,
    shared_adata_1=shared_adata_1,
    shared_adata_2=shared_adata_2,
    all_adata_1=all_adata_1,
    all_adata_2=all_adata_2,
    n_sample_1=1000,
    n_sample_2=1200
)

labels_1, labels_2 = getLabels(
    dataset_name=dataset_name,
    adata_1=shared_adata_1,
    adata_2=shared_adata_2
)

AnnData object with n_obs × n_vars = 10000 × 180
    obs: 'celltype.l1', 'celltype.l2' AnnData object with n_obs × n_vars = 10000 × 180
    obs: 'celltype.l1', 'celltype.l2' AnnData object with n_obs × n_vars = 10000 × 20729
    obs: 'celltype.l1', 'celltype.l2' AnnData object with n_obs × n_vars = 10000 × 224
    obs: 'celltype.l1', 'celltype.l2'


In [3]:
MMI = MMIHCL(
    shared_array_1=shared_adata_1.X,
    shared_array_2=shared_adata_2.X,
    all_array_1=all_adata_1.X,
    all_array_2=all_adata_2.X
)

MMI.setMatchingParameters(
    n_intermediate_match=1,
    n_final_match=1,
    verbose=True
)

MMI.constructGraph(
    n_neighbors_1=15,
    n_neighbors_2=15,
    n_components_1=30,
    n_components_2=30,
    verbose=True
)

We will perform one-to-one matching between two datasets.
Calculating adaptively k-neighbor adjacent matrices...
Two adaptively k-neighbor adjacent matrices have been calculated.
Calculating GCN aggregate matrices...
Two GCN aggregate matrices have been calculation calculated.


In [4]:
MMI.findInitialMatching(
    hyperegde_dim=32,
    n_epochs=1000,
    n_layers=2,
    learn_rate=2e-2,
    weight_decay=1e-3,
    temp=0.1,
    min_dist=1e-7,
    threshold=0.1,
    verbose=True
)

print(evalAccuracy(MMI.init_matching, labels_1, labels_2))

Learning shared feature embeddings...
Initializing the model and optimizer...
Fitting the model...
The hypergraph-based embedding learning is complete.
Finding the initial matching...
Filtering the initial matching...
The initial matching has been found and filtered.
0.9


In [5]:
MMI.CCAProjection(MMI.init_matching)

MMI.shared_array_1, MMI.shared_array_2 = MMI.all_array_cca_1.astype(np.float32), MMI.all_array_cca_2.astype(np.float32)

MMI.findInitialMatching(
    hyperegde_dim=32,
    n_epochs=1000,
    n_layers=2,
    learn_rate=2e-2,
    weight_decay=1e-3,
    temp=0.1,
    min_dist=1e-7,
    threshold=0.1,
    verbose=True
)

print(evalAccuracy(MMI.init_matching, labels_1, labels_2))

Learning shared feature embeddings...
Initializing the model and optimizer...
Fitting the model...
The hypergraph-based embedding learning is complete.
Finding the initial matching...
Filtering the initial matching...
The initial matching has been found and filtered.
0.896


In [6]:
# MMI.findRefinedMatchingCCA(
#     n_components=20,
#     max_iter=100,
#     n_iter=1,
#     min_dist=1e-7
# )

# print(evalAccuracy(MMI.refined_matching, labels_1, labels_2))

In [7]:
# protein_cca, rna_cca = MMI.all_array_cca_1, MMI.all_array_cca_2

# evalFOSCTTM(protein_cca, rna_cca)

# cca_adata = ad.AnnData(
#     np.concatenate((protein_cca, rna_cca), axis=0), 
#     dtype=np.float32
# )
# cca_adata.obs['datatype'] = ['protein'] * protein_cca.shape[0] + ['rna'] * rna_cca.shape[0]
# cca_adata.obs['celltype'] = labels_1 + labels_2

# np.savetxt('./MOIHCL_Cytof_human_H1N1.csv', protein_cca, delimiter=',', fmt='%.8f')
# np.savetxt('./MOIHCL_Cytof_human_IFNG.csv', rna_cca, delimiter=',', fmt='%.8f')

# PROTEIN_damn = pd.read_csv('./MOIHCL_Cytof_human_H1N1.csv', header=None, index_col=None).to_numpy().astype(np.float32)

# print(protein_cca.shape, rna_cca.shape)

# protein_cca, PROTEIN_damn

In [8]:
# cca_adata = ad.AnnData(
#     np.concatenate((protein_cca, rna_cca), axis=0), 
#     dtype=np.float32
# )
# cca_adata.obs['datatype'] = ['protein'] * protein_cca.shape[0] + ['RNA'] * rna_cca.shape[0]
# cca_adata.obs['celltype'] = labels_1 + labels_2

# sc.pp.neighbors(cca_adata, n_neighbors=15)
# sc.tl.umap(cca_adata)
# sc.pl.umap(cca_adata, color='datatype', size=25)
# sc.pl.umap(cca_adata, color='celltype', size=25)

In [9]:
# gc = evalGraphConnectivity(cca_adata.X, cca_adata.obs['celltype'])
# asw = evalAvgSilhouetteWidth(cca_adata.X, cca_adata.obs['celltype'])

# gc, asw

In [10]:
# MOI.findRefinedMatchingCCA(
#     n_components=20,
#     max_iter=1000,
#     n_iter=3,
#     min_dist=1e-7
# )

# print(evalAccuracy(MOI.refined_matching, labels_1, labels_2))

# MOI.filterBadMatching(MOI.refined_matching, 0.3)
# print(evalAccuracy(MOI.final_matching, labels_1, labels_2))

In [11]:
# shared_adata_1_sub = shared_adata_1[:1000, :]

# sc.pp.neighbors(shared_adata_1_sub, use_rep='X', n_neighbors=15)
# sc.tl.umap(shared_adata_1_sub)
# sc.pl.umap(shared_adata_1_sub, color='celltype.l1')